In [1]:
# !pip install -q hashstash[rec] stanza
import sys
sys.path.append('..')
from osp import *

In [2]:
get_nlp()

In [3]:
# !unzip -n osp_slices_1000.zip
# !mkdir -p /root/.cache/hashstash
# !rm -rf /root/.cache/hashstash/osp_slices_1000
# !mv osp_slices_1000 /root/.cache/hashstash

In [4]:
input_stash = STASH_SLICES
input_stash

Config,Param,Value
Path,Root Dir,/Users/rj416/github/ordinary-style-philosophy/data/raw/stash/osp_slices_1000
,Filename,data.db
Engine,Engine,lmdb
,Serializer,hashstash
,Compress,lz4
,B64,True
Stats,Len,89897


In [5]:
output_stash = STASH_SLICES_NLP
output_stash

Config,Param,Value
Path,Root Dir,/Users/rj416/github/ordinary-style-philosophy/data/raw/stash/osp_slices_1000_nlp
,Filename,data.db
Engine,Engine,lmdb
,Serializer,hashstash
,Compress,lz4
,B64,True
Stats,Len,33667


In [6]:
def get_slice_key(text_id, slice_id):
    return f'{text_id}__{int(slice_id):02d}'

In [7]:
slices_by_period = defaultdict(list)
df_meta = get_corpus_metadata()

for text_id,row in tqdm(list(df_meta.query('discipline=="Other"').iterrows())):
    text_slices = get_text_slices(text_id)
    for slice_id, slice_txt in text_slices.items():
        slices_by_period[row.period].append((get_slice_key(text_id, slice_id), slice_txt))

for k in slices_by_period:
    print(k, len(slices_by_period[k]))

100%|██████████| 32277/32277 [00:02<00:00, 15221.78it/s]

1975-2000 10373
2000-2025 15402
1950-1975 6679
1925-1950 3966
1900-1925 1587


In [8]:
inps = [x for k,v in slices_by_period.items() for x in random.sample(v,1000)]
len(inps)

5000

In [9]:
inps[0]

('other/10.2307/524189__01',
 "The Functioning and Effects of the Kenya Literacy Program G. Carron Background Over the last few decades, an increasing number of developing countries have embarked upon the organization of nationwide literacy programs.\nAlthough the rationale behind those programs varies from country to country, it is generally expected that efforts to increase the literacy levels of adults will have positive consequences for both the learners and the nation as a whole.\nIt is worth noting, however, that until now empirical evidence which could support these expectations has remained extremely weak.\nBy and large, adult literacy has been a neglected area in terms of data collection and research.\nIn many countries it may even be difficult to find precise information about simple facts such as the number of adults enrolled in literacy classes, the number of instructors, or the number of literacy proficiency certificates which have been delivered.\nIn most cases the illite

In [10]:
import random
random.shuffle(inps)

In [11]:
def gen_nlp_doc(txt, key, force=False, stash=output_stash):
    if force or key not in stash:
        nlp = get_nlp()
        doc = nlp(txt)
        stash[key] = doc.to_serialized()
        return doc

In [12]:
for key, txt in tqdm(inps):
    gen_nlp_doc(txt, key)

 10%|▉         | 497/5000 [39:58<6:02:08,  4.83s/it] 


KeyboardInterrupt: 